# 04 - Whitepaper Figures
Generate all publication-quality charts and the summary report.

In [22]:
import sys
sys.path.insert(0, '..')
import logging
logging.basicConfig(level=logging.INFO)
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.facecolor'] = 'white'

In [23]:
import datetime
import pickle
from pathlib import Path

from src.polymarket.market_discovery import load_discovered_markets
from src.freight.scraper import fetch_all_freight_indexes
from src.freight.normalize import prepare_freight_panel
from src.analysis.events import EventDetector
from src.analysis.correlation import CorrelationAnalyser, event_study
from src.analysis.impact_mapper import ImpactMapper

PANEL_PATH = Path('data/processed/timeseries_panel.csv')
ANALYSIS_CACHE = Path('data/processed/analysis_cache.pkl')

# ── Markets ───────────────────────────────────────────────────────────────────
markets_df = load_discovered_markets()

# ── Timeseries: load from panel CSV (built in NB 02) ─────────────────────────
if PANEL_PATH.exists():
    panel = pd.read_csv(PANEL_PATH, parse_dates=['date'])
    timeseries = {mid: grp.reset_index(drop=True) for mid, grp in panel.groupby('market_id')}
    print(f'Timeseries: {len(timeseries)} markets from cache')
else:
    raise FileNotFoundError(
        'timeseries_panel.csv not found. Run 02_data_collection.ipynb first.'
    )

# ── Freight data ──────────────────────────────────────────────────────────────
freight_raw = fetch_all_freight_indexes(use_synthetic_fallback=True)
freight_data = prepare_freight_panel(freight_raw)

# ── Analysis results: load from cache (built in NB 03) ───────────────────────
detector = EventDetector()
analyser = CorrelationAnalyser()
mapper = ImpactMapper()

if ANALYSIS_CACHE.exists():
    with open(ANALYSIS_CACHE, 'rb') as f:
        _cache = pickle.load(f)
    all_events = _cache['all_events']
    xcorr_results = _cache['xcorr_results']
    granger_results = _cache['granger_results']
    print(f'Analysis: loaded from cache ({len(all_events)} events, {len(xcorr_results)} xcorr pairs)')
else:
    # Fallback: recompute (run NB 03 first to avoid this)
    print('Analysis cache not found — recomputing (run NB 03 first to cache results).')
    all_events = detector.detect_all(timeseries, markets_df)
    xcorr_results = analyser.run_cross_correlations(timeseries, freight_data, markets_df)
    granger_results = analyser.run_granger_tests(timeseries, freight_data, markets_df)
    with open(ANALYSIS_CACHE, 'wb') as f:
        pickle.dump({
            'all_events': all_events,
            'xcorr_results': xcorr_results,
            'granger_results': granger_results,
        }, f)

events_df = detector.to_dataframe(all_events)
xcorr_df = analyser.xcorr_to_dataframe(xcorr_results)
granger_df = analyser.granger_to_dataframe(granger_results)
assessments = mapper.generate_assessments(all_events, markets_df, xcorr_results)

print(f'Ready. Markets: {len(timeseries)}, Events: {len(all_events)}, XCorr: {len(xcorr_results)}')

INFO:src.freight.scraper:Fetching freight index: BDI
INFO:src.freight.scraper:Attempting BDI download from Stooq …


Timeseries: 5322 markets from cache


INFO:src.freight.scraper:Attempting BDI download from TradingEconomics …
INFO:src.freight.scraper:Generated 560 synthetic BDI observations.
INFO:src.freight.scraper:  → 560 observations for BDI
INFO:src.freight.scraper:Fetching freight index: FBX_GLOBAL
INFO:src.freight.scraper:Generated 112 synthetic FBX_GLOBAL observations.
INFO:src.freight.scraper:  → 112 observations for FBX_GLOBAL
INFO:src.freight.scraper:Fetching freight index: FBX01
INFO:src.freight.scraper:Generated 112 synthetic FBX01 observations.
INFO:src.freight.scraper:  → 112 observations for FBX01
INFO:src.freight.scraper:Fetching freight index: FBX03
INFO:src.freight.scraper:Generated 112 synthetic FBX03 observations.
INFO:src.freight.scraper:  → 112 observations for FBX03
INFO:src.freight.scraper:Fetching freight index: FBX11
INFO:src.freight.scraper:Generated 112 synthetic FBX11 observations.
INFO:src.freight.scraper:  → 112 observations for FBX11
INFO:src.freight.normalize:Normalising freight index: BDI
INFO:src.frei

Analysis: loaded from cache (3614 events, 0 xcorr pairs)


INFO:src.analysis.impact_mapper:Generated 2893 impact assessments.


Ready. Markets: 5322, Events: 3614, XCorr: 0


## Chart 1: Dual-axis Overlay (Hero Chart)

In [24]:
from src.visualization.charts import (
    plot_dual_axis_overlay,
    plot_cross_correlation,
    plot_event_study,
    plot_correlation_heatmap,
    plot_annotated_timeline,
)

if xcorr_results:
    best = max(xcorr_results, key=lambda r: abs(r.peak_correlation))
    poly_df = timeseries.get(best.market_id)
    freight_df = freight_data.get(best.freight_index)
    mkt_events = (
        events_df[events_df['market_id'].astype(str) == str(best.market_id)]
        if not events_df.empty else pd.DataFrame()
    )
    event_ts = (
        list(pd.to_datetime(mkt_events['timestamp']))
        if not mkt_events.empty else []
    )
    if poly_df is not None and freight_df is not None:
        fig = plot_dual_axis_overlay(
            poly_df, freight_df,
            best.market_title, best.freight_index,
            event_timestamps=event_ts,
            filename_stem='fig1_dual_axis_hero',
        )
        plt.show()

## Chart 2: Cross-Correlation Plot

In [25]:
if xcorr_results:
    best = max(xcorr_results, key=lambda r: abs(r.peak_correlation))
    fig = plot_cross_correlation(
        best.lags, best.correlations, best.p_values,
        best.market_title, best.freight_index,
        best.peak_lag, best.peak_correlation,
        filename_stem='fig2_xcorr_best',
    )
    plt.show()
    print(f'Interpretation: {best.interpretation}')

## Chart 3: Event Study

In [15]:
event_study_results = []
for r in xcorr_results[:3]:
    poly_df = timeseries.get(r.market_id)
    freight_df = freight_data.get(r.freight_index)
    if poly_df is None or freight_df is None:
        continue
    mkt_events = (
        events_df[events_df['market_id'].astype(str) == str(r.market_id)]
        if not events_df.empty else pd.DataFrame()
    )
    if mkt_events.empty:
        continue
    event_ts_list = list(pd.to_datetime(mkt_events['timestamp']))
    es = event_study(
        r.market_id, r.market_title,
        event_ts_list, freight_df, r.freight_index,
    )
    if es is not None:
        event_study_results.append(es)
        fig = plot_event_study(
            es.event_window, es.mean_freight_change,
            es.ci_lower, es.ci_upper,
            es.baseline_change,
            es.market_title, es.freight_index, es.n_events,
            filename_stem=f'fig3_event_study_{r.market_id[:8]}',
        )
        plt.show()
        print(f'CAR (cumulative abnormal return): {es.cumulative_abnormal_return:.2f}%')

## Chart 4: Correlation Heatmap

In [16]:
if not xcorr_df.empty:
    fig = plot_correlation_heatmap(xcorr_df, filename_stem='fig4_heatmap')
    plt.show()

## Chart 5: Annotated Timeline

In [17]:
if xcorr_results and not events_df.empty:
    best = xcorr_results[0]
    poly_df = timeseries.get(best.market_id)
    freight_df = freight_data.get(best.freight_index)
    mkt_events = events_df[events_df['market_id'].astype(str) == str(best.market_id)]
    if poly_df is not None and freight_df is not None and not mkt_events.empty:
        fig = plot_annotated_timeline(
            poly_df, freight_df, mkt_events,
            best.market_title, best.freight_index,
            filename_stem='fig5_timeline',
        )
        plt.show()

## Generate Summary Report

In [21]:
from pathlib import Path
from datetime import datetime

report_lines = []
report_lines.append('# Polymarket Supply Chain Intelligence - Analysis Report')
report_lines.append(f'*Generated: {datetime.now().strftime("%Y-%m-%d %H:%M UTC")}*')
report_lines.append('---')
report_lines.append('## Executive Summary')
report_lines.append(f'- **Markets discovered:** {len(markets_df)}')
report_lines.append(f"- **Active markets:** {(markets_df['status'] == 'active').sum()}")
report_lines.append(f"- **Historical (closed) markets:** {(markets_df['status'] == 'closed').sum()}")
report_lines.append(f'- **Markets with timeseries data:** {len(timeseries)}')
report_lines.append(f'- **Significant probability shift events detected:** {len(all_events)}')
report_lines.append(f'- **Market-freight pairings analysed:** {len(xcorr_results)}')
report_lines.append(f"- **Statistically significant correlations (p < 0.05):** {xcorr_df['is_significant'].sum() if not xcorr_df.empty else 0}")
report_lines.append(f'- **Granger causality tests run:** {len(granger_results)}')
report_lines.append(f"- **Significant Granger results:** {granger_df['is_significant'].sum() if not granger_df.empty else 0}")
report_lines.append('---')
report_lines.append('## Key Findings')
report_lines.append('### Cross-Correlation Results')

if not xcorr_df.empty:
    for _, row in xcorr_df.sort_values('peak_correlation', ascending=False).head(5).iterrows():
        leads = 'Polymarket LEADS' if row['polymarket_leads'] else 'Freight leads'
        sig = 'significant' if row['is_significant'] else 'not significant'
        report_lines.append(f"- **{row['market_title'][:50]}** x {row['freight_index']}: r={row['peak_correlation']:.3f}, lag={row['peak_lag_days']}d ({leads}, {sig})")

report_lines.append('---')
report_lines.append('## Granger Causality')

if not granger_df.empty:
    for _, row in granger_df.sort_values('min_p_value').head(5).iterrows():
        sig = 'SIGNIFICANT' if row['is_significant'] else 'not significant'
        report_lines.append(f"- **{row['market_title'][:50]}** -> {row['freight_index']}: p={row['min_p_value']:.4f} at lag={row['best_lag']}d ({sig})")

report_lines.append('---')
report_lines.append(mapper.generate_report_section(assessments, top_n=5))
report_lines.append('---')
report_lines.append('## Figures Generated')

figures_dir = Path('../output/figures')
if figures_dir.exists():
    for fig_file in sorted(figures_dir.glob('*.png')):
        report_lines.append(f'- {fig_file.name}')

report_lines.append('---')
report_lines.append('## Caveats and Limitations')
report_lines.append('1. **Synthetic freight data**: Where live freight data was unavailable, synthetic data was generated for pipeline validation. Final analysis requires real BDI/FBX data.')
report_lines.append('2. **Correlation != causation**: Statistical relationships documented here are consistent with the leading indicator thesis but require further validation.')
report_lines.append('3. **Sample size**: Some pairings have limited overlapping observations, reducing statistical power.')
report_lines.append('4. **Market selection bias**: Markets were selected by keyword matching; some relevant markets may be missed.')
report_lines.append('5. **Data gaps**: Some Polymarket timeseries have gaps that could affect correlation estimates.')
report_lines.append('---')
report_lines.append('*Report generated by Polymarket SCM Intelligence MVP pipeline.*')

report_text = '\n'.join(report_lines)
report_path = Path('../output/report.md')
report_path.parent.mkdir(exist_ok=True)
report_path.write_text(report_text, encoding='utf-8')
print(f'Report saved to {report_path.resolve()}')
print('Report preview (first 2000 chars):')
print(report_text[:2000])

Report saved to C:\Users\kekoi\Projects\operations_research\polymarket-scm-intel\output\report.md
Report preview (first 2000 chars):
# Polymarket Supply Chain Intelligence - Analysis Report
*Generated: 2026-02-21 19:18 UTC*
---
## Executive Summary
- **Markets discovered:** 35912
- **Active markets:** 12889
- **Historical (closed) markets:** 23023
- **Markets with timeseries data:** 5322
- **Significant probability shift events detected:** 3614
- **Market-freight pairings analysed:** 0
- **Statistically significant correlations (p < 0.05):** 0
- **Granger causality tests run:** 0
- **Significant Granger results:** 0
---
## Key Findings
### Cross-Correlation Results
---
## Granger Causality
---
## Supply Chain Intelligence Alerts

*Generated 2026-02-21 19:18 UTC*

Top 5 highest-impact signals:

### Alert #1 — Confidence: HIGH
**Signal:** 521462: probability rose from 2% to 99% (+97pp shift)
**Date:** 2025-05-02
**Category:** tariffs_us_china
**Impact Score:** 0.1958

**Affected Routes:*

## Complete
All whitepaper figures and the summary report have been generated.

Figures are saved in output/figures/ (PNG + SVG). Report is at output/report.md.